In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))

from xau_ml import features as F, sim as S
from xau_ml.config import (
    HORIZONS, LABEL_SETS, MARGIN_GRID, TAU_GRID, MIN_VAL_TRADES,
    ModelParams, WF2_BURN_IN, WF2_FIRST_FOLD, WF2_LAST_FOLD, TRAIN,
    MODEL_DIR,
)
from xau_ml.cv import Fold, monthly_folds, assert_folds, uniqueness
from xau_ml.data import load
from xau_ml.labels import add_labels
from xau_ml.model import Calibrator, fit_lgbm
from xau_ml.v2 import SIDES, REDUCED_BASE, EXTRA_COLS, baseline_trades
from xau_ml.v3 import add_v3_features, V3_REDUCED_PLUS

print("imports ok")

## 1. Load data + build features (same as v3/v4)

In [ ]:
hz_name = "intraday"
hz = HORIZONS[hz_name]
ls = LABEL_SETS["ct25"]
mp = ModelParams()

# Load broker CSVs per timeframe
from xau_ml.data import load as load_tf
bars = {hz.tf: load_tf(hz.tf)}
for tf in hz.htf:
    bars[tf] = load_tf(tf)

fr = F.build(bars[hz.tf], hz.tf, {tf: bars[tf] for tf in hz.htf})
fr = F.add_extras(fr, bars["1d"])
fr = add_v3_features(fr)  # calendar + market features
fr = add_labels(fr, hz, ls)

print(f"Frame: {len(fr)} bars, {fr['date'].min()} to {fr['date'].max()}")
print(f"Bull win rate: {fr['y_bull'].mean():.3f}, Bear win rate: {fr['y_bear'].mean():.3f}")

## 2. Regime features

Same composite regime from v4: SMA20/50/200 trend + price position + momentum = score -3..+3, bucketed to UP(+1)/DOWN(-1)/SIDE(0).

In [ ]:
def add_regime_features(fr, d1):
    """Composite regime from daily SMA20/50/200 — merged backward like other HTF context."""
    c = d1["close"].to_numpy()
    h, low = d1["high"].to_numpy(), d1["low"].to_numpy()
    sma20 = pd.Series(c).rolling(20).mean().to_numpy()
    sma50 = pd.Series(c).rolling(50).mean().to_numpy()
    sma200 = pd.Series(c).rolling(200).mean().to_numpy()
    a14 = F.atr(h, low, c, 14)

    trend_long = np.where(sma50 > sma200, 1, np.where(sma50 < sma200, -1, 0))
    price_vs_50 = np.where(c > sma50, 1, -1)
    mom = np.where(sma20 > sma50, 1, np.where(sma20 < sma50, -1, 0))
    score = trend_long + price_vs_50 + mom
    regime = np.where(score >= 2, 1, np.where(score <= -2, -1, 0))

    with np.errstate(divide="ignore", invalid="ignore"):
        trend_str = np.where(a14 > 0, (c - sma50) / a14, 0.0)
        slope = np.where(a14 > 0, (sma50 - np.r_[np.full(5, np.nan), sma50[:-5]]) / a14, 0.0)

    feat = pd.DataFrame({
        "regime": regime.astype(float),
        "regime_trend_str": trend_str,
        "regime_sma50_slope": slope,
        "d1_t_close_regime": d1["t_close"].to_numpy(),
    }, index=d1.index)

    left = fr[["t_close"]].reset_index()
    merged = pd.merge_asof(
        left.sort_values("t_close"),
        feat.sort_values("d1_t_close_regime"),
        left_on="t_close",
        right_on="d1_t_close_regime",
        direction="backward",
    ).set_index("index")

    out = fr.copy()
    for c in ["regime", "regime_trend_str", "regime_sma50_slope"]:
        out[c] = merged.loc[out.index, c].to_numpy()
    return out

REGIME_COLS = ["regime", "regime_trend_str", "regime_sma50_slope"]

fr = add_regime_features(fr, bars["1d"])
print("Regime distribution in training window (2025-01 to 2025-07):")
train_mask = (fr["date"] >= "2025-01-01") & (fr["date"] <= "2025-07-31")
print(fr.loc[train_mask, "regime"].value_counts().sort_index().rename({-1: "DOWN", 0: "SIDE", 1: "UP"}))

## 3. Walk-forward folds + feature columns

In [ ]:
# Walk-forward folds: expanding monthly, same as v2/v3
folds = monthly_folds(fr, hz, WF2_FIRST_FOLD, WF2_LAST_FOLD, train_from=TRAIN[0])
assert_folds(fr, folds, hz)
fold_names = [f.name for f in folds]

# Feature columns: v3 reduced + regime
all_cols = F.feature_columns(fr)
base_reduced = [c for c in all_cols if c.startswith("d1_") or c in REDUCED_BASE]
# Level-1 specialist features: base reduced + session + calendar + market + regime
from xau_ml.v3 import V3_SESSION_COLS, V3_CALENDAR, V3_MARKET
L1_COLS = base_reduced + [c for c in (V3_SESSION_COLS + V3_CALENDAR + V3_MARKET + REGIME_COLS) if c in all_cols]
L1_COLS = list(dict.fromkeys(L1_COLS))  # deduplicate preserving order

# Meta-model context features (NOT the raw price features — just the context)
META_CONTEXT_COLS = [c for c in ["regime", "regime_trend_str", "regime_sma50_slope",
                                  "atr_regime", "adx14", "session",
                                  "d1_adx14", "d1_rsi14", "d1_trend",
                                  "vix_level", "dxy_dist_sma20"] if c in all_cols]

print(f"Folds: {len(folds)} ({fold_names[0]} to {fold_names[-1]})")
print(f"Level-1 features: {len(L1_COLS)}")
print(f"Meta context features: {len(META_CONTEXT_COLS)} — {META_CONTEXT_COLS}")

## 4. Level-1: Specialist models with class balancing

Each specialist LightGBM is trained with `scale_pos_weight` to correct its class imbalance:
- **Bull specialist**: bull wins ~32% → `scale_pos_weight` ~2.1
- **Bear specialist**: bear wins ~18% → `scale_pos_weight` ~4.6

This forces each model to treat its minority class (winning trades) as more important, producing better-calibrated probabilities for its specific task.

In [ ]:
def fit_specialist(X, y, hz, mp, w=None):
    """LightGBM binary classifier with scale_pos_weight for class balancing."""
    params = {**mp.params}
    pos_rate = y.mean()
    if 0 < pos_rate < 1:
        params["scale_pos_weight"] = (1 - pos_rate) / pos_rate
    return fit_lgbm(X, y, hz, mp, w=w, extra_params={"scale_pos_weight": params["scale_pos_weight"]})


def run_specialists_oof(fr, folds, cols, hz, mp):
    """Walk-forward OOF predictions from both specialist models.

    Returns a DataFrame with columns:
      pos, fold, p_bull_raw, p_bear_raw, plus per-fold ML metrics.
    Each specialist is trained ONLY on its own target with scale_pos_weight.
    """
    parts, metric_rows = [], []
    for f in folds:
        X_train = fr.iloc[f.train][cols]
        X_test = fr.iloc[f.test][cols]

        preds = {}
        for side in SIDES:
            y_train = fr[f"y_{side}"].to_numpy()[f.train].astype(int)
            w = uniqueness(fr, f.train, f"exit_{side}")
            booster = fit_specialist(X_train, y_train, hz, mp, w=w)
            preds[side] = booster.predict(X_test)

        parts.append(pd.DataFrame({
            "pos": f.test, "fold": f.name,
            "p_bull_raw": preds["bull"],
            "p_bear_raw": preds["bear"],
        }))

        rec = {"fold": f.name, "train_rows": len(f.train), "test_rows": len(f.test)}
        for side in SIDES:
            y_test = fr[f"y_{side}"].to_numpy()[f.test]
            rec[f"auc_{side}"] = roc_auc_score(y_test, preds[side]) if 0 < y_test.mean() < 1 else np.nan
            rec[f"logloss_{side}"] = log_loss(y_test, np.clip(preds[side], 1e-6, 1-1e-6), labels=[0,1])
        metric_rows.append(rec)

    return pd.concat(parts, ignore_index=True), pd.DataFrame(metric_rows)


oof, ml_metrics = run_specialists_oof(fr, folds, L1_COLS, hz, mp)
print("=== Level-1 specialist OOF metrics (mean across folds) ===")
print(ml_metrics[[c for c in ml_metrics.columns if c.startswith("auc") or c.startswith("logloss")]].mean().to_string())
print(f"\nOOF rows: {len(oof)}")

## 5. Level-2: Meta-model (3-class: FLAT / LONG / SHORT)

The meta-model sees the OOF predictions from both specialists **plus** market context features. It learns:
- When both specialists agree on direction → take the trade
- When the regime conflicts with the signal → stay flat
- When bear probability is high enough → go short (something Level-1 alone couldn't decide)

**Label construction**: `y_meta = 1 (LONG) if bull wins, 2 (SHORT) if bear wins, 0 (FLAT) otherwise`

Training uses nested OOF: fold k's meta-model trains on the specialist OOF of folds < k only (no leakage).

In [ ]:
def build_meta_features(oof_df, fr, context_cols):
    """Build meta-model input: specialist OOF probs + context features from the frame."""
    pos = oof_df["pos"].to_numpy()
    meta = pd.DataFrame({
        "p_bull": oof_df["p_bull_raw"].to_numpy(),
        "p_bear": oof_df["p_bear_raw"].to_numpy(),
        "p_diff": oof_df["p_bull_raw"].to_numpy() - oof_df["p_bear_raw"].to_numpy(),
        "p_max": np.maximum(oof_df["p_bull_raw"].to_numpy(), oof_df["p_bear_raw"].to_numpy()),
        "p_sum": oof_df["p_bull_raw"].to_numpy() + oof_df["p_bear_raw"].to_numpy(),
    }, index=oof_df.index)
    # Add context features from the original frame
    for c in context_cols:
        meta[c] = fr[c].to_numpy()[pos]
    return meta


def build_meta_labels(fr, pos):
    """3-class: 0=FLAT (neither wins), 1=LONG (bull wins), 2=SHORT (bear wins)."""
    y_bull = fr["y_bull"].to_numpy()[pos].astype(int)
    y_bear = fr["y_bear"].to_numpy()[pos].astype(int)
    # If both win (rare / impossible with triple barrier), prefer bull
    y_meta = np.where(y_bull == 1, 1, np.where(y_bear == 1, 2, 0))
    return y_meta


# Build meta features + labels for all OOF rows
meta_X = build_meta_features(oof, fr, META_CONTEXT_COLS)
meta_y = build_meta_labels(fr, oof["pos"].to_numpy())

print(f"Meta features: {list(meta_X.columns)}")
print(f"\nMeta label distribution:")
for lbl, name in [(0, "FLAT"), (1, "LONG"), (2, "SHORT")]:
    print(f"  {name}: {(meta_y == lbl).sum()} ({(meta_y == lbl).mean():.1%})")

## 6. Nested walk-forward: full stacked ensemble

For each fold k (scored from burn-in onward):
1. Level-1 specialist OOF predictions are already computed above
2. Meta-model trains on OOF of folds < k (nested, no leakage)
3. Meta-model predicts fold k → signal (LONG/SHORT/FLAT)

We compare two approaches:
- **Exp A (v4 baseline)**: specialist + tau/margin signal rule (same as v3/v4)
- **Exp B (meta-model)**: specialist + LightGBM meta-model for signal decision
- **Exp C (meta-model + confidence filter)**: meta-model with probability threshold

In [ ]:
META_MP = ModelParams(params={
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 8,        # small: meta-model should be simple
    "max_depth": 3,
    "min_child_samples": 80,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 10.0,
    "verbose": -1,
    "seed": 42,
    "deterministic": True,
    "num_threads": 4,
}, max_rounds=500, early_stopping=50)


def nested_ensemble_trading(fr, oof, meta_X, meta_y, fold_names, burn_in, min_conf=0.0):
    """Nested walk-forward with meta-model deciding LONG/SHORT/FLAT.

    For fold k:
      - Meta-model trains on specialist OOF of folds 0..k-1
      - Predicts fold k's meta features → class probs
      - Signal: argmax of [FLAT, LONG, SHORT] if max_prob > min_conf, else FLAT
    """
    sig = np.zeros(len(oof), dtype="int8")
    meta_probs = np.zeros((len(oof), 3))
    log = []

    for k, name in enumerate(fold_names):
        cur = (oof["fold"] == name).to_numpy()
        prior = oof["fold"].isin(fold_names[:k]).to_numpy()

        if k < burn_in or not prior.any():
            continue

        # Train meta-model on prior folds' OOF
        X_meta_train = meta_X.loc[prior]
        y_meta_train = meta_y[prior]

        # Use class weights for the 3-class imbalance
        meta_booster = lgb.train(
            {**META_MP.params},
            lgb.Dataset(X_meta_train, y_meta_train, free_raw_data=False),
            num_boost_round=META_MP.max_rounds,
        )

        # Predict on current fold
        X_meta_test = meta_X.loc[cur]
        probs = meta_booster.predict(X_meta_test)  # (n, 3): [flat, long, short]
        meta_probs[cur] = probs

        # Signal from meta-model
        pred_class = probs.argmax(axis=1)
        max_prob = probs.max(axis=1)

        # class 0=FLAT, 1=LONG, 2=SHORT → signal: 0, +1, -1
        fold_sig = np.where(pred_class == 1, 1, np.where(pred_class == 2, -1, 0)).astype("int8")

        # Confidence filter
        if min_conf > 0:
            fold_sig[max_prob < min_conf] = 0

        sig[cur] = fold_sig

        log.append({
            "fold": name,
            "train_meta_rows": int(prior.sum()),
            "signals": int((fold_sig != 0).sum()),
            "long_signals": int((fold_sig == 1).sum()),
            "short_signals": int((fold_sig == -1).sum()),
            "flat_share": float((fold_sig == 0).mean()),
            "mean_long_prob": float(probs[:, 1].mean()),
            "mean_short_prob": float(probs[:, 2].mean()),
        })

    scored = oof["fold"].isin(fold_names[burn_in:]).to_numpy()
    trades = S.run(fr, oof["pos"].to_numpy()[scored], sig[scored])
    return trades, pd.DataFrame(log), sig, scored, meta_probs


# --- Exp A: specialist + tau/margin (v4 baseline) ---
from xau_ml.v2 import nested_trading as nested_trading_v2
trades_A, log_A, cal_A, sig_A, scored_A = nested_trading_v2(
    fr, oof, fold_names, WF2_BURN_IN, MIN_VAL_TRADES[hz.tf]
)

# --- Exp B: specialist + meta-model (no confidence filter) ---
trades_B, log_B, sig_B, scored_B, probs_B = nested_ensemble_trading(
    fr, oof, meta_X, meta_y, fold_names, WF2_BURN_IN, min_conf=0.0
)

# --- Exp C: specialist + meta-model (confidence > 0.40) ---
trades_C, log_C, sig_C, scored_C, probs_C = nested_ensemble_trading(
    fr, oof, meta_X, meta_y, fold_names, WF2_BURN_IN, min_conf=0.40
)

# --- Exp D: specialist + meta-model (confidence > 0.50) ---
trades_D, log_D, sig_D, scored_D, probs_D = nested_ensemble_trading(
    fr, oof, meta_X, meta_y, fold_names, WF2_BURN_IN, min_conf=0.50
)

print("Walk-forward complete.")

## 7. Walk-forward results comparison

In [ ]:
results = {
    "A: specialist + tau/m": trades_A,
    "B: meta-model (no filter)": trades_B,
    "C: meta-model (conf>0.40)": trades_C,
    "D: meta-model (conf>0.50)": trades_D,
}

rows = []
for name, tr in results.items():
    mt = S.metrics(tr)
    rows.append({"experiment": name, **mt})

wf_summary = pd.DataFrame(rows).set_index("experiment")
print("=== Walk-forward trading results (scored folds only) ===")
print(wf_summary[["trades", "win_rate", "profit_factor", "expectancy_r", "total_r", "max_dd_pct", "longs", "shorts"]].to_string())
print()

# Show long/short balance
for name, tr in results.items():
    if len(tr):
        side_counts = tr["side"].value_counts()
        long_pct = side_counts.get("long", 0) / len(tr) * 100
        print(f"{name}: {long_pct:.0f}% long, {100-long_pct:.0f}% short")

In [ ]:
# Per-fold detail for meta-model experiments
print("=== Exp B: meta-model per-fold signals ===")
print(log_B[["fold", "signals", "long_signals", "short_signals", "flat_share"]].to_string(index=False))
print()
print("=== Exp C: meta-model (conf>0.40) per-fold signals ===")
print(log_C[["fold", "signals", "long_signals", "short_signals", "flat_share"]].to_string(index=False))

## 8. Aug-Sep 2026 holdout backtest

The real test: how does the stacked ensemble perform on completely unseen data?

In [ ]:
from xau_ml import holdout as H

# Fetch holdout bars
hbars = H.load_bars()
print(f"Holdout 15m: {len(hbars['15m'])} bars, {hbars['15m']['date'].min()} to {hbars['15m']['date'].max()}")
print(f"Holdout 1h:  {len(hbars['1h'])} bars")
print(f"Holdout 1d:  {len(hbars['1d'])} bars")

In [ ]:
# Build holdout frame with all features (same pipeline as training)
htfs = {tf: hbars[tf] for tf in hz.htf}
hfr = F.build(hbars[hz.tf], hz.tf, htfs)
hfr = F.add_extras(hfr, hbars["1d"])
hfr = add_v3_features(hfr, use_cache=True)
hfr = add_labels(hfr, hz, ls)
hfr = add_regime_features(hfr, hbars["1d"])

hpos = H.holdout_rows(hfr)
print(f"Holdout decision rows: {len(hpos)}")
print(f"Holdout regime distribution:")
print(hfr.iloc[hpos]["regime"].value_counts().sort_index().rename({-1.0: "DOWN", 0.0: "SIDE", 1.0: "UP"}))

In [ ]:
# Train final Level-1 specialists on ALL available training data (all folds)
all_train = np.concatenate([f.train for f in folds])
X_all_train = fr.iloc[all_train][L1_COLS]

specialists = {}
for side in SIDES:
    y_all = fr[f"y_{side}"].to_numpy()[all_train].astype(int)
    w_all = uniqueness(fr, all_train, f"exit_{side}")
    specialists[side] = fit_specialist(X_all_train, y_all, hz, mp, w=w_all)
    pos_rate = y_all.mean()
    print(f"{side} specialist: pos_rate={pos_rate:.3f}, scale_pos_weight={(1-pos_rate)/pos_rate:.2f}")

# Level-1 predictions on holdout
h_X = hfr.iloc[hpos][L1_COLS]
h_p_bull = specialists["bull"].predict(h_X)
h_p_bear = specialists["bear"].predict(h_X)

print(f"\nHoldout specialist predictions:")
print(f"  p_bull: mean={h_p_bull.mean():.3f}, p>0.5: {(h_p_bull>0.5).sum()}")
print(f"  p_bear: mean={h_p_bear.mean():.3f}, p>0.5: {(h_p_bear>0.5).sum()}")

In [ ]:
# Train final meta-model on ALL OOF data (from scored folds)
meta_X_all = meta_X.copy()
meta_y_all = meta_y.copy()

meta_booster_final = lgb.train(
    {**META_MP.params},
    lgb.Dataset(meta_X_all, meta_y_all, free_raw_data=False),
    num_boost_round=META_MP.max_rounds,
)

# Build holdout meta features
h_meta = pd.DataFrame({
    "p_bull": h_p_bull,
    "p_bear": h_p_bear,
    "p_diff": h_p_bull - h_p_bear,
    "p_max": np.maximum(h_p_bull, h_p_bear),
    "p_sum": h_p_bull + h_p_bear,
})
for c in META_CONTEXT_COLS:
    h_meta[c] = hfr[c].to_numpy()[hpos]

# Meta-model predictions on holdout
h_meta_probs = meta_booster_final.predict(h_meta)
h_pred_class = h_meta_probs.argmax(axis=1)
h_max_prob = h_meta_probs.max(axis=1)

print("Holdout meta-model class predictions:")
for cls, name in [(0, "FLAT"), (1, "LONG"), (2, "SHORT")]:
    print(f"  {name}: {(h_pred_class == cls).sum()} ({(h_pred_class == cls).mean():.1%})")

# Signals for each experiment
# Exp A: tau/margin with Platt calibration (use all OOF to fit calibrator + tau/m)
all_oof_bull = oof["p_bull_raw"].to_numpy()
all_oof_bear = oof["p_bear_raw"].to_numpy()
all_oof_pos = oof["pos"].to_numpy()
cal_bull = Calibrator.fit("platt", all_oof_bull, fr["y_bull"].to_numpy()[all_oof_pos])
cal_bear = Calibrator.fit("platt", all_oof_bear, fr["y_bear"].to_numpy()[all_oof_pos])
h_p_bull_cal = cal_bull(h_p_bull)
h_p_bear_cal = cal_bear(h_p_bear)
_, best_tm = S.select_tau_m(fr, all_oof_pos, cal_bull(all_oof_bull), cal_bear(all_oof_bear),
                             TAU_GRID, MARGIN_GRID, MIN_VAL_TRADES[hz.tf])
print(f"\nExp A best tau/m: tau={best_tm['tau']}, m={best_tm['m']}")
h_sig_A = S.signals(h_p_bull_cal, h_p_bear_cal, best_tm["tau"], best_tm["m"])

# Exp B: meta-model (no filter)
h_sig_B = np.where(h_pred_class == 1, 1, np.where(h_pred_class == 2, -1, 0)).astype("int8")

# Exp C: meta-model (conf > 0.40)
h_sig_C = h_sig_B.copy()
h_sig_C[h_max_prob < 0.40] = 0

# Exp D: meta-model (conf > 0.50)
h_sig_D = h_sig_B.copy()
h_sig_D[h_max_prob < 0.50] = 0

print(f"\nHoldout signals:")
for name, sig in [("A: tau/m", h_sig_A), ("B: meta", h_sig_B), ("C: meta>0.40", h_sig_C), ("D: meta>0.50", h_sig_D)]:
    print(f"  {name}: long={int((sig==1).sum())}, short={int((sig==-1).sum())}, flat={int((sig==0).sum())}")

In [ ]:
# Run holdout trades + baselines
holdout_results = {
    "A: specialist + tau/m": S.run(hfr, hpos, h_sig_A),
    "B: meta-model": S.run(hfr, hpos, h_sig_B),
    "C: meta (conf>0.40)": S.run(hfr, hpos, h_sig_C),
    "D: meta (conf>0.50)": S.run(hfr, hpos, h_sig_D),
    "always_long": baseline_trades(hfr, hpos, "always_long"),
    "always_short": baseline_trades(hfr, hpos, "always_short"),
}

holdout_summary = H.summary(holdout_results)
print("=== HOLDOUT Aug-Sep 2026 ===")
print(holdout_summary[["trades", "win_rate", "profit_factor", "expectancy_r", "total_r", "max_dd_pct", "longs", "shorts"]].to_string())
print()

# Long/short balance on holdout
for name, tr in holdout_results.items():
    if len(tr) and name not in ("always_long", "always_short"):
        side_counts = tr["side"].value_counts()
        long_n = side_counts.get("long", 0)
        short_n = side_counts.get("short", 0)
        total = long_n + short_n
        if total > 0:
            print(f"{name}: {long_n}L / {short_n}S ({long_n/total*100:.0f}% / {short_n/total*100:.0f}%)")

In [ ]:
# Monthly breakdown for the best meta-model experiment
print("=== Holdout by month: Exp B (meta-model) ===")
if len(holdout_results["B: meta-model"]):
    print(H.by_month(holdout_results["B: meta-model"]).to_string())
print()
print("=== Holdout by month: Exp C (meta conf>0.40) ===")
if len(holdout_results["C: meta (conf>0.40)"]):
    print(H.by_month(holdout_results["C: meta (conf>0.40)"]).to_string())

## 9. Meta-model feature importance

Which features does the meta-model rely on most to decide long/short/flat?

In [ ]:
imp = pd.DataFrame({
    "feature": meta_booster_final.feature_name(),
    "importance": meta_booster_final.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
imp["pct"] = imp["importance"] / imp["importance"].sum() * 100
print("Meta-model feature importance (gain):")
print(imp.to_string(index=False))

## 10. Swing horizon (repeat for 1h bars)

Same architecture applied to the swing horizon for completeness.

In [ ]:
hz_sw = HORIZONS["swing"]
ls_sw = LABEL_SETS["ct25"]

bars_sw = {hz_sw.tf: load_tf(hz_sw.tf)}
for tf in hz_sw.htf:
    bars_sw[tf] = load_tf(tf)

fr_sw = F.build(bars_sw[hz_sw.tf], hz_sw.tf, {tf: bars_sw[tf] for tf in hz_sw.htf})
fr_sw = F.add_extras(fr_sw, bars_sw["1d"])
fr_sw = add_v3_features(fr_sw)
fr_sw = add_labels(fr_sw, hz_sw, ls_sw)
fr_sw = add_regime_features(fr_sw, bars_sw["1d"])

folds_sw = monthly_folds(fr_sw, hz_sw, WF2_FIRST_FOLD, WF2_LAST_FOLD, train_from=TRAIN[0])
assert_folds(fr_sw, folds_sw, hz_sw)
fold_names_sw = [f.name for f in folds_sw]

all_cols_sw = F.feature_columns(fr_sw)
base_reduced_sw = [c for c in all_cols_sw if c.startswith("d1_") or c in REDUCED_BASE]
L1_COLS_SW = base_reduced_sw + [c for c in (V3_SESSION_COLS + V3_CALENDAR + V3_MARKET + REGIME_COLS) if c in all_cols_sw]
L1_COLS_SW = list(dict.fromkeys(L1_COLS_SW))

META_CTX_SW = [c for c in META_CONTEXT_COLS if c in all_cols_sw]

print(f"Swing frame: {len(fr_sw)} bars")
print(f"Swing folds: {len(folds_sw)}")

# Level-1
oof_sw, ml_sw = run_specialists_oof(fr_sw, folds_sw, L1_COLS_SW, hz_sw, mp)
meta_X_sw = build_meta_features(oof_sw, fr_sw, META_CTX_SW)
meta_y_sw = build_meta_labels(fr_sw, oof_sw["pos"].to_numpy())

# Exp A (tau/m) + Exp B (meta) + Exp C (meta>0.40)
trades_A_sw, log_A_sw, _, _, _ = nested_trading_v2(fr_sw, oof_sw, fold_names_sw, WF2_BURN_IN, MIN_VAL_TRADES[hz_sw.tf])
trades_B_sw, log_B_sw, _, _, _ = nested_ensemble_trading(fr_sw, oof_sw, meta_X_sw, meta_y_sw, fold_names_sw, WF2_BURN_IN, 0.0)
trades_C_sw, log_C_sw, _, _, _ = nested_ensemble_trading(fr_sw, oof_sw, meta_X_sw, meta_y_sw, fold_names_sw, WF2_BURN_IN, 0.40)

sw_results = {
    "A: tau/m": trades_A_sw,
    "B: meta": trades_B_sw,
    "C: meta>0.40": trades_C_sw,
}
sw_rows = [{"exp": n, **S.metrics(t)} for n, t in sw_results.items()]
print("\n=== Swing walk-forward ===")
print(pd.DataFrame(sw_rows).set_index("exp")[["trades", "win_rate", "profit_factor", "expectancy_r", "total_r", "longs", "shorts"]].to_string())

## 11. Swing holdout

In [ ]:
# Swing holdout
htfs_sw = {tf: hbars[tf] for tf in hz_sw.htf}
hfr_sw = F.build(hbars[hz_sw.tf], hz_sw.tf, htfs_sw)
hfr_sw = F.add_extras(hfr_sw, hbars["1d"])
hfr_sw = add_v3_features(hfr_sw, use_cache=True)
hfr_sw = add_labels(hfr_sw, hz_sw, ls_sw)
hfr_sw = add_regime_features(hfr_sw, hbars["1d"])

hpos_sw = H.holdout_rows(hfr_sw)
print(f"Swing holdout rows: {len(hpos_sw)}")

# Train final specialists + meta on all swing folds
all_train_sw = np.concatenate([f.train for f in folds_sw])
specs_sw = {}
for side in SIDES:
    y_s = fr_sw[f"y_{side}"].to_numpy()[all_train_sw].astype(int)
    w_s = uniqueness(fr_sw, all_train_sw, f"exit_{side}")
    specs_sw[side] = fit_specialist(fr_sw.iloc[all_train_sw][L1_COLS_SW], y_s, hz_sw, mp, w=w_s)

h_pb_sw = specs_sw["bull"].predict(hfr_sw.iloc[hpos_sw][L1_COLS_SW])
h_pr_sw = specs_sw["bear"].predict(hfr_sw.iloc[hpos_sw][L1_COLS_SW])

# Meta-model
meta_final_sw = lgb.train({**META_MP.params}, lgb.Dataset(meta_X_sw, meta_y_sw), num_boost_round=META_MP.max_rounds)
h_meta_sw = pd.DataFrame({"p_bull": h_pb_sw, "p_bear": h_pr_sw, "p_diff": h_pb_sw-h_pr_sw,
                           "p_max": np.maximum(h_pb_sw, h_pr_sw), "p_sum": h_pb_sw+h_pr_sw})
for c in META_CTX_SW:
    h_meta_sw[c] = hfr_sw[c].to_numpy()[hpos_sw]
h_mp_sw = meta_final_sw.predict(h_meta_sw)
h_cls_sw = h_mp_sw.argmax(axis=1)
h_conf_sw = h_mp_sw.max(axis=1)

# Signals
cal_b_sw = Calibrator.fit("platt", oof_sw["p_bull_raw"].to_numpy(), fr_sw["y_bull"].to_numpy()[oof_sw["pos"].to_numpy()])
cal_r_sw = Calibrator.fit("platt", oof_sw["p_bear_raw"].to_numpy(), fr_sw["y_bear"].to_numpy()[oof_sw["pos"].to_numpy()])
_, best_sw = S.select_tau_m(fr_sw, oof_sw["pos"].to_numpy(), cal_b_sw(oof_sw["p_bull_raw"].to_numpy()),
                             cal_r_sw(oof_sw["p_bear_raw"].to_numpy()), TAU_GRID, MARGIN_GRID, MIN_VAL_TRADES[hz_sw.tf])

h_sig_A_sw = S.signals(cal_b_sw(h_pb_sw), cal_r_sw(h_pr_sw), best_sw["tau"], best_sw["m"]) if best_sw else np.zeros(len(hpos_sw), dtype="int8")
h_sig_B_sw = np.where(h_cls_sw==1, 1, np.where(h_cls_sw==2, -1, 0)).astype("int8")
h_sig_C_sw = h_sig_B_sw.copy(); h_sig_C_sw[h_conf_sw < 0.40] = 0

sw_holdout = {
    "A: tau/m": S.run(hfr_sw, hpos_sw, h_sig_A_sw),
    "B: meta": S.run(hfr_sw, hpos_sw, h_sig_B_sw),
    "C: meta>0.40": S.run(hfr_sw, hpos_sw, h_sig_C_sw),
    "always_long": baseline_trades(hfr_sw, hpos_sw, "always_long"),
    "always_short": baseline_trades(hfr_sw, hpos_sw, "always_short"),
}
print("\n=== Swing HOLDOUT Aug-Sep 2026 ===")
print(H.summary(sw_holdout)[["trades", "win_rate", "profit_factor", "expectancy_r", "total_r", "longs", "shorts"]].to_string())

## 12. Conclusion

Compare the stacked ensemble approach vs the v4 baseline:

| Approach | Architecture | Signal decision |
|----------|-------------|----------------|
| v4 (Exp A) | Specialist bull + bear → Platt calibrate → tau/margin rule | Rule-based: p_bull >= tau AND p_bull - p_bear >= margin |
| v5 (Exp B) | Specialist bull + bear → Meta-model LightGBM 3-class | Learned: meta-model sees specialist outputs + context → LONG/SHORT/FLAT |
| v5 (Exp C) | Same as B + confidence filter | Learned + filtered: only trade when meta-model confidence > threshold |

The meta-model can learn **when to go short** from the context (regime, VIX, trend strength) even when the specialist alone wouldn't produce a strong bear signal. This is the key advantage over the tau/margin rule which is symmetric but can't adapt to regime.

# XAUUSD ML v0.5 — Stacked Ensemble
**Specialist Bull + Bear models → Meta-model (long/short/flat)**

v4 showed that a single model pair remains bull-biased (85% long trades) even with regime features and class balancing.
This notebook trains **separate specialist models** per side, each class-balanced, then combines them through a **meta-learner** that decides long/short/flat using OOF predictions + context.

## Architecture
```
Level 1:  Bull-specialist LightGBM (scale_pos_weight) → p_bull_oof
          Bear-specialist LightGBM (scale_pos_weight) → p_bear_oof

Level 2:  Meta-model (LightGBM 3-class) inputs:
            [p_bull, p_bear, p_bull-p_bear, regime, atr_regime, adx14, session, vix_level, ...]
          → class 0=FLAT, 1=LONG, 2=SHORT
```

## Key differences from v4
- Each specialist gets its own `scale_pos_weight` tuned to its class imbalance
- Meta-model is trained on OOF predictions (no leakage) + market context
- Meta-model learns WHEN to go flat vs take a directional signal
- Holdout: Aug-Sep 2026